# Benchmark offline (bez Hugging Face)

Ten sam układ co `Colab_Runner.ipynb`, ale **bez login()** i bez pobierania z internetu.

Wymaganie: model musi być już w `models_cache/` (pobierz raz przez notebook online).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/Steganography_colab'
MODEL_CACHE_DIR = f'{PROJECT_DIR}/models_cache'

os.chdir(PROJECT_DIR)
os.environ['MODEL_CACHE_DIR'] = MODEL_CACHE_DIR
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

print('cwd:', Path.cwd())
print('model cache:', MODEL_CACHE_DIR)

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
required = ['run_experiments.py', 'dummy_processor.py']
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje plikow w {PROJECT_DIR}: {missing}')
print('Wymagane pliki OK')

## Konfiguracja przebiegu

In [ ]:
# humaneval | perplexity | capacity | binoculars
TEST = 'humaneval'

# qwen | llama | gemma  (musi byc w cache)
MODEL = 'gemma'

# np. 0.01 albo 'all'
THRESHOLD = 0.0

TOP_N = 15
MAX_NEW_TOKENS = 256

# HumanEval only: None = wszystkie 164 | '5' | '0-10' | '0,3,7'
HUMANEVAL_TASKS = None

In [ ]:
!python run_experiments.py \
  --offline \
  --list-cached-models \
  --model-cache-dir {MODEL_CACHE_DIR}

In [ ]:
_humaneval_args = f' --humaneval-tasks {HUMANEVAL_TASKS}' if HUMANEVAL_TASKS else ''

!python run_experiments.py \
  --offline \
  --test {TEST} \
  --model {MODEL} \
  --threshold {THRESHOLD} \
  --top-n {TOP_N} \
  --max-new-tokens {MAX_NEW_TOKENS} \
  --model-cache-dir {MODEL_CACHE_DIR}{_humaneval_args}

In [ ]:
import csv

summary_csv = Path('results/summary.csv')
if summary_csv.exists():
    with summary_csv.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'Rows in summary.csv: {len(rows)}')
    for row in rows[-5:]:
        print(row)
else:
    print('summary.csv not found yet')